In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask



In [ ]:
# TO DO

import os
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split


images_dir = "/kaggle/input/q3-stage3-2026/dataset/images"
masks_dir  = "/kaggle/input/q3-stage3-2026/dataset/masks"


class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, file_names, img_tf=None, mask_tf=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.file_names = file_names
        self.img_tf = img_tf
        self.mask_tf = mask_tf

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        img_name = self.file_names[idx]
        base = os.path.splitext(img_name)[0]
        mask_name = base + ".png"

        # here you may face error because of the problem i said in q2
        img_path = os.path.join(self.images_dir, img_name)
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        if self.img_tf:
            image = self.img_tf(image)

        if self.mask_tf:
            mask = self.mask_tf(mask)

        mask = remap_mask(mask.squeeze(0))  #here is remap mask classes
        return image, mask


IMG_SIZE = (256, 256)

img_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
])

mask_transforms = transforms.Compose([
    transforms.Resize(IMG_SIZE, interpolation=transforms.InterpolationMode.NEAREST),
    transforms.PILToTensor(),
])

# here you may face error because of the problem i said in q2
all_images = sorted([f for f in os.listdir(images_dir) if f.endswith(".jpg")])

train_files, val_files = train_test_split(all_images, test_size=0.2, random_state=42)

train_dataset = SUIMDataset(images_dir, masks_dir, train_files, img_transforms, mask_transforms)
val_dataset   = SUIMDataset(images_dir, masks_dir, val_files,   img_transforms, mask_transforms)


train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=8, shuffle=False)


# Display some images and masks
for i in range(3):
    img, mask = train_dataset[i]

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(img.permute(1, 2, 0))
    axes[0].set_title("Image")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Mask")
    axes[1].axis("off")

    plt.show()


In [ ]:
# TO DO


!pip install -q segmentation-models-pytorch

import segmentation_models_pytorch as smp


NUM_CLASSES = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)


In [ ]:
# TO DO


from tqdm import tqdm

def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0

    for images, masks in tqdm(dataloader):
        images = images.to(device)
        masks = masks.long().to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.long().to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# TO DO

import torch.nn as nn


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 5 # i set epoches to 5 becasue if i write more it take too much time
train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
# TO DO

model.eval()

with torch.no_grad():
    for i in range(3):
        image, mask = val_dataset[i]

        input_tensor = image.unsqueeze(0).to(device)
        pred = model(input_tensor)
        pred_mask = torch.argmax(pred, dim=1).squeeze(0).cpu()

        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        axes[0].imshow(image.permute(1, 2, 0).cpu())
        axes[0].set_title("Image")
        axes[0].axis("off")

        axes[1].imshow(mask.cpu(), cmap="gray")
        axes[1].set_title("Ground Truth")
        axes[1].axis("off")

        axes[2].imshow(pred_mask, cmap="gray")
        axes[2].set_title("Prediction")
        axes[2].axis("off")

        plt.show()
